Utilizzando lo script della lezione, effettua un test comparativo per comprendere l'importanza del Learning Rate nel fine-tuning

* Configurazione A (Corretta): Esegui il fine-tuning con learning-rate = 1e-5 (come nel codice della lezione)
* Configurazione B (Errata): modifica il learning-rate del fine-tuning portandolo a 1e-2 (un valore molto alto)

Compito:
Osserva l'andamento dell accuracy durante le epoche della Fase 2 nella configurazione B. Spiega perchè la precisione crolla improvvisamente invece di migliorare

Suggerimento: utilizza la funzione keras.models.load_model per ricaricare il modello salvato dopo la fase 1, in modo da partire dalla stessa base per entrambi i test.

In [ ]:
import os

# 1. SET BACKEND
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve

# --- PREPARAZIONE DATI ---
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

def filter_binary(x, y, c1, c2):
    mask = (y == c1) | (y == c2)
    x_filtered = x[mask.flatten()]
    y_filtered = (y[mask.flatten()] == c2).astype(int) 
    return x_filtered, y_filtered

x_train, y_train = filter_binary(x_train, y_train, 3, 5)
x_test, y_test = filter_binary(x_test, y_test, 3, 5)

# --- COSTRUZIONE MODELLO ---
base_model = keras.applications.MobileNetV2(input_shape=(160, 160, 3), include_top=False, weights="imagenet")
base_model.trainable = False

inputs = keras.Input(shape=(32, 32, 3))
x = keras.layers.Resizing(160, 160)(inputs)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.2)(x)
outputs = keras.layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)

# --- FASE 1: FEATURE EXTRACTION (Head Training) ---
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
print("\n[INFO] Fase 1: Addestramento della testa...")
model.fit(x_train, y_train, epochs=3, validation_split=0.2, batch_size=32)

# SALVATAGGIO POST-FASE 1: Fondamentale per il test comparativo
model.save("head_trained.keras")

# --- CONFIGURAZIONE A: FINE-TUNING CORRETTO (LR = 1e-5) ---
print("\n" + "="*50)
print("[TEST A] Inizio Fine-Tuning con LR = 1e-5 (Corretto)")
model_a = keras.models.load_model("head_trained.keras")
# Sblocchiamo il backbone
model_a.layers[4].trainable = True # Il layer 4 è il base_model nel nostro grafo
for layer in model_a.layers[4].layers[:100]:
    layer.trainable = False

model_a.compile(optimizer=keras.optimizers.Adam(1e-5), loss="binary_crossentropy", metrics=["accuracy"])
history_a = model_a.fit(x_train, y_train, epochs=3, validation_split=0.2, batch_size=32)

# --- CONFIGURAZIONE B: FINE-TUNING ERRATO (LR = 1e-2) ---
print("\n" + "="*50)
print("[TEST B] Inizio Fine-Tuning con LR = 1e-2 (Errato)")
model_b = keras.models.load_model("head_trained.keras")
# Sblocchiamo il backbone allo stesso modo
model_b.layers[4].trainable = True
for layer in model_b.layers[4].layers[:100]:
    layer.trainable = False

model_b.compile(optimizer=keras.optimizers.Adam(1e-2), loss="binary_crossentropy", metrics=["accuracy"])
# OSSERVA QUI: L'accuratezza probabilmente crollerà o oscillerà violentemente
history_b = model_b.fit(x_train, y_train, epochs=3, validation_split=0.2, batch_size=32)

# --- VALUTAZIONE FINALE (Modello A) ---
y_pred_probs = model_a.predict(x_test).ravel()
fpr, tpr, _ = roc_curve(y_test, y_pred_probs)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, label=f'ROC A (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.title('Confronto ROC Post Fine-Tuning')
plt.legend()
plt.show()